In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="02-visual-search",
    notebook_name="04_interview_walkthrough.ipynb"
)

# Visual Search: Complete 45-Minute Interview Simulation

## What This Notebook Covers

This is a **fully scripted, 45-minute mock interview** for the visual search system design question at a staff ML engineer level. It covers:

1. Interview timeline — what happens when
2. Phase-by-phase interviewer/candidate dialogue with exact scripts
3. Whiteboard-style architecture diagrams
4. System design "power moves" — what separates staff from senior answers
5. Six follow-up questions with WEAK vs. STRONG answer comparisons
6. Scoring rubric with junior / senior / staff level indicators
7. 30-second elevator pitch
8. Key vocabulary cheat sheet

**How to use this notebook:** Read the scripts out loud. Practice the power moves until they feel natural. Time yourself.

---

## 1. Interview Timeline

A 45-minute ML system design interview follows a predictable structure. Understanding the timeline helps you **pace yourself** — a common mistake is spending 25 minutes on requirements and only 5 minutes on serving.

The ideal allocation:

| Phase | Minutes | What You Do |
|-------|---------|-------------|
| Clarifying questions | 5 min | Ask the right 4-5 questions |
| ML framing | 5 min | Define objective, input/output, ML category |
| Data & model | 15 min | Data pipeline, architecture, training, loss |
| Evaluation | 5 min | Offline (nDCG) and online (CTR) metrics |
| Serving | 10 min | Prediction + indexing pipelines, ANN, re-ranking |
| Follow-ups | 5 min | Deep dives chosen by interviewer |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ======================================================================
# Interview Timeline Visualization
# ======================================================================

phases = [
    ('Clarifying\nQuestions', 5, '#4FC3F7'),
    ('ML Framing', 5, '#81C784'),
    ('Data &\nModel', 15, '#FFB74D'),
    ('Evaluation', 5, '#CE93D8'),
    ('Serving\nArchitecture', 10, '#EF9A9A'),
    ('Follow-up\nQ&A', 5, '#80CBC4'),
]

fig, ax = plt.subplots(figsize=(14, 5))

cumulative = 0
bar_height = 0.5
y = 0.35

for (name, duration, color) in phases:
    ax.barh(y, duration, left=cumulative, height=bar_height, color=color,
            edgecolor='white', linewidth=2)
    # Label inside bar if wide enough, else above
    label_x = cumulative + duration / 2
    if duration >= 8:
        ax.text(label_x, y, f'{name}\n{duration} min', ha='center', va='center',
                fontsize=10, fontweight='bold')
    else:
        ax.text(label_x, y + 0.35, f'{name}\n{duration}m', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
    
    # Tick mark at boundary
    ax.axvline(cumulative, color='#666', linewidth=1, linestyle='--', alpha=0.5)
    cumulative += duration

ax.axvline(cumulative, color='#666', linewidth=1, linestyle='--', alpha=0.5)

# Danger zones
ax.axvspan(0, 10, alpha=0.03, color='green')
ax.axvspan(35, 45, alpha=0.06, color='red')
ax.text(43, 0.05, 'Clock\nrunning\nout\!', ha='center', fontsize=9, color='#C62828', style='italic')

ax.set_xlim(0, 45)
ax.set_ylim(-0.1, 1.0)
ax.set_xlabel('Minutes into Interview', fontsize=13)
ax.set_title('45-Minute ML System Design Interview Timeline\nVisual Search Edition',
             fontsize=15, fontweight='bold')
ax.set_yticks([])

# Time markers
for t in range(0, 46, 5):
    ax.text(t, -0.05, f'{t}m', ha='center', va='top', fontsize=9, color='#555')

# Legend
patches = [mpatches.Patch(facecolor=c, label=f'{n.replace(chr(10), " ")} ({d}min)')
           for (n, d, c) in phases]
ax.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.2),
          ncol=3, fontsize=9)

plt.tight_layout()
plt.savefig('interview_timeline.png', dpi=130, bbox_inches='tight')
plt.show()

print("Key pacing rules:")
print("  - Clarifying questions: NEVER skip, but don't spend > 6 minutes")
print("  - ML framing: Clear and crisp — 2-3 sentences max per point")
print("  - Data & Model: This is the meat — give concrete architecture details")
print("  - Evaluation: State nDCG confidently, explain why in 30 seconds")
print("  - Serving: ANN + re-ranking is what separates senior/staff")
print("  - Follow-ups: Listen for what the interviewer cares about most")

## 2. Phase-by-Phase Dialogue Scripts

### Phase 1: Clarifying Questions (0-5 min)

This is where you establish **shared understanding** before diving into solutions. The worst thing you can do is spend 30 minutes designing the wrong system.

---

**INTERVIEWER:** "Design a visual search system — like Pinterest's Lens feature. Users can take a photo and find visually similar images."

---

**CANDIDATE:** *(takes 20 seconds to jot down notes, then looks up)*

"Before I dive in, I have a few clarifying questions to make sure I'm solving the right problem.

First — should results be **ranked** by visual similarity, or is this more of a filter where we just return a set of matching images?"

**INTERVIEWER:** "Yes, ranked. Most similar first."

**CANDIDATE:** "Great. Are we supporting **image-only queries**, or should users also be able to search with text — like 'red sneakers' combined with a photo?"

**INTERVIEWER:** "Image only for now. You can mention text-to-image as a future extension."

**CANDIDATE:** "Understood. Should results be **personalized** to the user, or does the same query image always return the same results for everyone?"

**INTERVIEWER:** "Not personalized for now."

**CANDIDATE:** "What's the approximate **scale**? How many images are in the corpus?"

**INTERVIEWER:** "100 to 200 billion images."

**CANDIDATE:** "That's an important constraint — it immediately tells me we need approximate nearest neighbor search rather than exact. One last question: what **user actions** do we have available for training data? Do we have clicks, purchases, saves?"

**INTERVIEWER:** "We have impressions and clicks."

**CANDIDATE:** *(nods)* "Perfect. Let me summarize: we're designing a **ranked visual similarity retrieval system**, image-only, non-personalized, at 200 billion images, with click and impression data available for training. I'll proceed with that scope."

---

**Why this works:** The candidate asked exactly 5 questions, summarized the answers, and framed the scope in a single crisp sentence. They also previewed a constraint (ANN needed at 200B scale), signaling deep technical knowledge immediately.

### Phase 2: ML Framing (5-10 min)

---

**CANDIDATE:** "Let me frame this as an ML problem before touching any architecture.

The ML objective is: **accurately retrieve images that are visually similar to a query image**, where 'similar' is defined by visual appearance — shape, color, texture, style.

This is a **ranking problem** solved through **representation learning**. The key insight is that we do not train a model that directly scores image pairs — that would be O(N²) at inference time and completely infeasible. Instead, we train an encoder that maps every image to a fixed-dimensional **embedding vector**, such that similar images have nearby embeddings in this vector space.

At inference time:
```
Input:  query image
Output: embedding vector (e.g., 256-D)
Search: find images with the closest embeddings in the index
Return: ranked list of similar images
```

The embedding space approach decouples the **hard offline work** (embedding all 200 billion images) from the **fast online search** (finding nearest vectors to the query embedding).

I'll use a **contrastive learning** approach to train this encoder — I can walk through the details in the Data & Model section."

---

**INTERVIEWER:** "How is this different from just doing classification?"

**CANDIDATE:** "Great question. Classification assigns images to discrete classes — it does not produce a continuous similarity score between arbitrary image pairs. More importantly, classification requires defining all classes upfront. A visual search system needs to handle open-world similarity — two images of the same unusual lamp should be similar, even if 'unusual lamp' is not a class we trained on. Representation learning via contrastive training produces embeddings that generalize to unseen visual concepts."

---

**Power Move:** The candidate explicitly mentioned the decoupling of offline indexing from online search. This shows they're already thinking about the serving architecture.

### Phase 3: Data & Model (10-25 min)

---

**CANDIDATE:** "Let me walk through data, model architecture, and training.

**Data.** We have three tables: images with metadata, users with demographics, and user-image interactions — impressions and clicks. For visual search, we are **not** using metadata or user features. We train purely on pixel data using self-supervision.

**Preprocessing.** Every image goes through four steps: resize to 224×224, convert to float and scale to [0,1], z-score normalize using ImageNet channel statistics, and ensure RGB color mode. This is the same pipeline for both the indexing and prediction paths — critical for consistency.

**Model architecture.** I'd use a **ResNet-50 backbone** pretrained on ImageNet, with the classification head replaced by a **projection head** — two fully connected layers mapping 2048 dimensions to 256 dimensions. The output is **L2-normalized** to put all embeddings on a unit hypersphere. This makes cosine similarity equal to a simple dot product, which is extremely efficient to compute.

For scale, I'd consider **ViT-B/16** for better representation quality, but ResNet-50 is a proven starting point with excellent speed-accuracy tradeoff.

**Training.** I'd use **SimCLR-style self-supervised contrastive learning**. We generate two augmented views of each image — random crops, color jitter, grayscale, blur — and train the model to maximize similarity between views of the same image while minimizing similarity to all other images in the batch.

The loss is **NT-Xent** — normalized temperature-scaled cross-entropy. The temperature parameter tau (typically 0.07) controls how sharply the model distinguishes positives from negatives.

**Positive pair sourcing strategy.** I'd start with self-supervision because it is free and scales to any dataset size. Once the base model is trained, I'd fine-tune on click data — if a user clicked image B after searching with image A, those are a positive pair. Click data introduces noise (~20-30% of clicks are not truly visually similar), but provides real-world signal.

For fine-tuning, I'd use **differential learning rates** — lower LR for the pre-trained backbone to preserve features, higher LR for the projection head."

---

**INTERVIEWER:** "Why L2 normalization? Why not just use raw embeddings?"

**CANDIDATE:** "Two reasons. First, L2 normalization makes cosine similarity equal to dot product — and dot products are cheap at scale, especially with SIMD/GPU optimizations. Second, raw embeddings have different magnitudes for different images, which would make similarity scores incomparable across images. L2 normalization puts every embedding on the same unit hypersphere, so all similarity scores are in [-1, 1] and are directly comparable." 

### Phase 4: Evaluation (25-30 min)

---

**CANDIDATE:** "For evaluation, I'd use both offline and online metrics.

**Offline.** Our primary metric is **nDCG** — Normalized Discounted Cumulative Gain. Here is why it is the right choice: it handles continuous relevance scores (our evaluation set labels each candidate 0-5, not just relevant/irrelevant), it penalizes putting a highly relevant image at position 5 instead of position 1, and it normalizes against the ideal ranking so scores are between 0 and 1 and comparable across queries.

I would specifically report nDCG@10 as the headline number, since users rarely look past the first 10 results.

Metrics I would *not* use as primary: Precision@k because it doesn't reward ranking; Recall@k because the denominator is undefined for visual similarity (how many 'dog photos' exist in 200 billion images?); MRR because it only considers the first relevant result.

**Online.** Once deployed, the primary metrics are **click-through rate** on the returned results, and **time spent** on images surfaced by visual search (weekly and monthly). We measure these via A/B testing — 5% of traffic on the new model, 95% on baseline, with statistical significance testing at p < 0.05."

---

**INTERVIEWER:** "How would you detect a failure mode where the model is just memorizing training images?"

**CANDIDATE:** "Two checks. First, I'd evaluate on a held-out set with no overlap with training queries — if nDCG is high on training-similar queries but much lower on novel queries, that's a sign of overfitting. Second, I'd look at the **uniformity** of the embedding space — a model that is memorizing will produce embeddings that cluster tightly around training images, leaving large empty regions of the hypersphere. I'd measure this with the Wang & Isola uniformity metric: log mean exp(-2 * pairwise_distance). A good model has high uniformity (embeddings spread out) and high alignment (positive pairs close together)." 

### Phase 5: Serving Architecture (30-40 min)

---

**CANDIDATE:** "There are two distinct pipelines: prediction and indexing. Let me walk through both.

**Prediction pipeline** (real-time, < 100ms SLA):
1. **Preprocessing service** — resize, normalize, ensure RGB. Runs on CPU, ~1ms.
2. **Embedding generation service** — runs the ResNet-50 on GPU, ~15ms for a single image.
3. **ANN search service** — queries the FAISS index. ~10ms for nprobe=128, D=256.
4. **Re-ranking service** — applies business rules: remove private/deleted images, deduplicate near-identical results, enforce owner diversity. Returns top 50 from 500 ANN candidates.

**Indexing pipeline** (offline/batch):
1. All 200 billion images are processed through the same preprocessing + embedding pipeline in batch on a GPU fleet.
2. We run k-means with nlist=65,536 clusters to learn IVF centroids.
3. We build the FAISS IVFFlat index, shard it across servers.
4. When new images are uploaded, they are added incrementally — no full rebuild needed.

**The ANN index I'd use:** FAISS IVFFlat as the baseline. The key parameter is nprobe — at nprobe=128 out of 65,536 clusters, we get ~95% Recall@10 with ~10ms query latency. This is the recall-vs-speed knob.

**Memory optimization:** With float32 embeddings, 200B × 256 × 4 bytes = 200 TB. Using FAISS IVFFlat with product quantization (PQ32x8), we compress to ~6 TB — a 32× reduction — which fits on roughly 12 servers with 512 GB RAM each.

**For even more scale or better accuracy:** I'd consider HNSW (Hierarchical Navigable Small World) graphs for better recall-latency tradeoff, or ScaNN (Google's library) which adds anisotropic quantization for higher precision at the same memory budget."

---

**Power move:** The candidate mentioned PQ compression numbers without being asked. They also offered two upgrade paths (HNSW, ScaNN), signaling breadth of knowledge.

## 3. Whiteboard-Style Architecture Diagrams

The following code generates the two canonical architecture diagrams you should be able to draw on a whiteboard (or Excalidraw) during the interview. Practice drawing these freehand.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

# ======================================================================
# Two canonical architecture diagrams for the visual search interview
# ======================================================================

fig, axes = plt.subplots(2, 1, figsize=(18, 16))

# ============================================================
# TOP: Prediction Pipeline
# ============================================================
ax = axes[0]
ax.set_xlim(0, 11)
ax.set_ylim(0, 5)
ax.set_title('Prediction Pipeline (Real-Time, < 100ms SLA)', 
             fontsize=16, fontweight='bold', pad=12)
ax.axis('off')

pred_boxes = [
    (0.1, 1.7, 'User Query\nImage', '#FFE0B2', 1.4, '~0ms'),
    (1.8, 1.7, 'Preprocessing\nService\n(CPU)', '#B3E5FC', 1.6, '~1ms'),
    (3.7, 1.7, 'Embedding\nGeneration\n(GPU: ResNet-50)', '#C8E6C9', 1.8, '~15ms'),
    (5.8, 1.7, 'ANN Search\nService\n(FAISS IVFFlat)', '#F8BBD0', 1.8, '~10ms'),
    (7.9, 1.7, 'Re-ranking\nService\n(Business Rules)', '#D1C4E9', 1.8, '~5ms'),
    (10.0, 1.7, 'Top-50\nResults', '#E8F5E9', 1.2, ''),
]

prev_end = None
for (x, y, text, color, w, timing) in pred_boxes:
    box = FancyBboxPatch((x, y), w, 1.3, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + 0.65, text, ha='center', va='center',
            fontsize=9, fontweight='bold', multialignment='center')
    if timing:
        ax.text(x + w/2, y - 0.25, timing, ha='center', fontsize=9, 
                color='#1565C0', style='italic')
    if prev_end is not None:
        ax.annotate('', xy=(x, y+0.65), xytext=(prev_end, y+0.65),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))
    prev_end = x + w

# FAISS index (looked up by ANN service)
idx_box = FancyBboxPatch((5.4, 0.1), 2.4, 0.9, boxstyle='round,pad=0.08',
                         facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(idx_box)
ax.text(6.6, 0.55, 'FAISS Index\n(200B embeddings, PQ32x8)', 
        ha='center', va='center', fontsize=9, fontweight='bold')
ax.annotate('', xy=(6.8, 1.7), xytext=(6.8, 1.0),
            arrowprops=dict(arrowstyle='->', color='#F57F17', lw=2, linestyle='dashed'))
ax.text(6.4, 1.4, 'nprobe=128', ha='center', fontsize=8, color='#F57F17')

# Total time
ax.text(5.5, 4.5, 'Total end-to-end: ~35ms compute + ~20ms network = well under 100ms SLA',
        ha='center', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.9, edgecolor='#4CAF50', linewidth=2))

# ============================================================
# BOTTOM: Indexing Pipeline
# ============================================================
ax2 = axes[1]
ax2.set_xlim(0, 11)
ax2.set_ylim(0, 5)
ax2.set_title('Indexing Pipeline (Offline + Incremental Updates)', 
              fontsize=16, fontweight='bold', pad=12)
ax2.axis('off')

# Batch path
batch_y = 2.8
batch_boxes = [
    (0.1, batch_y, 'All 200B\nImages\n(S3)', '#FFE0B2', 1.6),
    (2.0, batch_y, 'Distributed\nPreprocessing\n(1K CPU cores)', '#B3E5FC', 1.8),
    (4.2, batch_y, 'GPU Fleet\nEmbedding\n(1K A100s)', '#C8E6C9', 1.8),
    (6.4, batch_y, 'Distributed\nk-means\n(nlist=65536)', '#F8BBD0', 1.8),
    (8.6, batch_y, 'Build FAISS\nIndex Shards\n(PQ32x8)', '#D1C4E9', 1.8),
]

prev_end = None
for (x, y, text, color, w) in batch_boxes:
    box = FancyBboxPatch((x, y), w, 1.3, boxstyle='round,pad=0.08',
                         facecolor=color, edgecolor='#333', linewidth=2)
    ax2.add_patch(box)
    ax2.text(x + w/2, y + 0.65, text, ha='center', va='center',
             fontsize=9, fontweight='bold', multialignment='center')
    if prev_end is not None:
        ax2.annotate('', xy=(x, y+0.65), xytext=(prev_end, y+0.65),
                     arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))
    prev_end = x + w

# Incremental path
inc_y = 0.9
inc_boxes = [
    (0.1, inc_y, 'New Image\nUploaded', '#FFCCBC', 1.6),
    (2.0, inc_y, 'Kafka\nQueue', '#B3E5FC', 1.4),
    (3.7, inc_y, 'Embedding\nWorker\n(GPU)', '#C8E6C9', 1.6),
    (5.6, inc_y, 'Assign to\nNearest IVF\nCluster', '#F8BBD0', 1.8),
    (7.7, inc_y, 'Update\nPosting List\n(< 5 min)', '#D1C4E9', 1.8),
]

prev_end = None
for (x, y, text, color, w) in inc_boxes:
    box = FancyBboxPatch((x, y), w, 1.3, boxstyle='round,pad=0.08',
                         facecolor=color, edgecolor='#4CAF50', linewidth=2)
    ax2.add_patch(box)
    ax2.text(x + w/2, y + 0.65, text, ha='center', va='center',
             fontsize=9, fontweight='bold', multialignment='center')
    if prev_end is not None:
        ax2.annotate('', xy=(x, y+0.65), xytext=(prev_end, y+0.65),
                     arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2.5))
    prev_end = x + w

# Labels
ax2.text(-0.05, batch_y + 1.35, 'BATCH\n(Nightly)', ha='left', fontsize=10,
         fontweight='bold', color='#1565C0',
         bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.8))
ax2.text(-0.05, inc_y + 1.35, 'INCR.\n(Real-time)', ha='left', fontsize=10,
         fontweight='bold', color='#2E7D32',
         bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.8))

ax2.text(5.5, 4.5, 
         'Batch rebuild: ~30min with 1,000 A100 GPUs  |  Incremental: searchable in < 5 minutes',
         ha='center', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#FFF9C4', alpha=0.9, edgecolor='#F57F17', linewidth=2))

plt.tight_layout()
plt.savefig('architecture_diagrams.png', dpi=130, bbox_inches='tight')
plt.show()

print("These are the two diagrams to practice drawing on a whiteboard.")
print("Practice drawing them in under 3 minutes each.")
print("The key insight: the two pipelines share the SAME model but serve different purposes.")

## 4. System Design Power Moves

### What Separates Staff from Senior Answers

At the senior level, the interviewer expects you to correctly design the system. At the **staff level**, they expect you to:

1. Identify non-obvious constraints and tradeoffs
2. Proactively mention failure modes and how to handle them
3. Quantify everything (latency, memory, cost)
4. Discuss the evolution of the system over time
5. Show awareness of neighboring problems (content moderation, personalization, etc.)

Below are the specific "power moves" — things to say that instantly signal staff-level thinking.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import textwrap

# ======================================================================
# Power Moves: Visual Cheat Sheet
# ======================================================================

power_moves = [
    {
        'topic': 'ANN vs Exact NN',
        'senior': 'Use FAISS for ANN search to handle scale.',
        'staff': 'Use FAISS IVFFlat with nlist=sqrt(N)=65536, nprobe=128 for 95% recall at ~10ms. '
                 'Use PQ32x8 to compress 200TB of float32 embeddings to 6TB. '
                 'HNSW is an alternative with better recall-latency curve but 5× higher memory.',
        'color': '#B3E5FC',
    },
    {
        'topic': 'Model Training',
        'senior': 'Use SimCLR with NT-Xent loss and data augmentation.',
        'staff': 'SimCLR needs very large batches (4096+) for enough negatives. '
                 'At limited GPU memory, use MoCo (momentum contrast) — it maintains a queue of '
                 'negatives so you get 65k negatives even with batch=256. '
                 'Use differential learning rates: 1e-5 for backbone, 1e-3 for projection head.',
        'color': '#C8E6C9',
    },
    {
        'topic': 'Index Freshness',
        'senior': 'Rebuild the index daily.',
        'staff': 'Nightly full rebuild + incremental updates for new uploads. '
                 'Two-phase consistency: new images assigned to existing IVF clusters (no k-means rerun), '
                 'searchable within 5 minutes. Full rebuild with updated model on a new index slot, '
                 'then atomic swap. Zero downtime. Model and index must always be in sync.',
        'color': '#F8BBD0',
    },
    {
        'topic': 'Re-ranking',
        'senior': 'Filter private images and near-duplicates.',
        'staff': 'Re-ranking is a multi-objective optimization. Order of filters matters: '
                 '(1) hard blockers (deleted, private, policy), '
                 '(2) dedup (cosine sim > 0.98), '
                 '(3) diversity (max 3 per owner), '
                 '(4) optional: personalization re-rank based on user history. '
                 'Mention positional bias: top results get more clicks regardless of quality.',
        'color': '#D1C4E9',
    },
    {
        'topic': 'Training Data Quality',
        'senior': 'Use self-supervision because it is free and scalable.',
        'staff': 'Self-supervision is the base; augmentation-based positives may not reflect '
                 'real visual similarity. Layer in click data with IPS (inverse propensity scoring) '
                 'to debias position effects. Hard negative mining (in-batch hardest negatives) '
                 'speeds convergence 3-5×. Consider domain-specific augmentations — e.g., '
                 'do NOT do aggressive color jitter for fashion (color IS the similarity signal).',
        'color': '#FFE0B2',
    },
    {
        'topic': 'Evaluation',
        'senior': 'Use nDCG offline and CTR online.',
        'staff': 'nDCG@10 is the offline headline, but also monitor embedding alignment + uniformity '
                 'as proxy for representation quality. Online: CTR can be gamed by showing '
                 'clickbait results — also track session depth (how many results users view) '
                 'and conversion rate (clicks that lead to saves/purchases). '
                 'A/B test with Bonferroni correction for multiple comparisons.',
        'color': '#FFF9C4',
    },
]

fig, axes = plt.subplots(len(power_moves), 1, figsize=(18, 24))

for i, (move, ax) in enumerate(zip(power_moves, axes)):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 2)
    ax.axis('off')
    
    # Background
    bg = mpatches.FancyBboxPatch((0, 0), 10, 2, boxstyle='round,pad=0.05',
                                  facecolor=move['color'], edgecolor='#999', linewidth=1.5, alpha=0.4)
    ax.add_patch(bg)
    
    # Topic header
    ax.text(0.15, 1.7, f"Topic: {move['topic']}", fontsize=12, fontweight='bold', va='top')
    
    # Senior answer
    ax.text(0.15, 1.35, 'SENIOR answer:', fontsize=10, fontweight='bold', color='#1565C0', va='top')
    wrapped_s = textwrap.fill(move['senior'], width=90)
    ax.text(0.15, 1.15, wrapped_s, fontsize=9, color='#1565C0', va='top', style='italic')
    
    # Staff answer
    ax.text(0.15, 0.82, 'STAFF answer (add this):', fontsize=10, fontweight='bold', color='#2E7D32', va='top')
    wrapped_p = textwrap.fill(move['staff'], width=90)
    ax.text(0.15, 0.65, wrapped_p, fontsize=9, color='#2E7D32', va='top')

fig.suptitle('Staff-Level Power Moves: What to Add Beyond the Senior Answer',
             fontsize=16, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig('power_moves.png', dpi=110, bbox_inches='tight')
plt.show()

print("Memorize the STAFF column for each topic.")
print("In the interview, lead with the senior answer, then immediately add the staff upgrade.")

## 5. Six Common Follow-up Questions: WEAK vs STRONG

The interviewer will probe deeper after your main design. These are the six most common follow-up questions for visual search, with explicit WEAK and STRONG answer comparisons.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import textwrap

# ======================================================================
# Follow-up Q&A: WEAK vs STRONG Answer Visualization
# ======================================================================

qna_pairs = [
    {
        'question': 'Q1: How would you handle a query image that is a close-up crop of a larger image?',
        'weak': 'We would preprocess it the same way as any other image -- resize to 224x224 and embed it.',
        'strong': (
            'The crop changes the composition significantly, which could hurt embedding quality. '
            'Three approaches: (1) Use a larger backbone (ViT-L with 16x16 patches) that captures '
            'fine-grained details. (2) Apply multi-scale inference: extract embeddings at the original '
            'crop, 2x zoom-out, and 4x zoom-out, then aggregate (average pooling). '
            '(3) Use an object detection model (e.g., YOLO) to first identify the main subject in the '
            'crop, normalize the bounding box, and embed the detected region. '
            'Pinterest uses region proposals (PinSage) for this reason.'
        ),
    },
    {
        'question': 'Q2: How would you handle content policy -- e.g., blocking nudity or hate speech in results?',
        'weak': 'We would add a filter in the re-ranking service to remove flagged images.',
        'strong': (
            'Content moderation happens at two levels. '
            'Upstream: images flagged by the content moderation system are excluded from the index '
            'entirely -- they never appear in any search results. This is safer than re-ranking filters '
            'because it prevents serving even before the ANN stage. '
            'Downstream: the re-ranking service checks a blocklist of recently flagged image IDs '
            '(since index updates lag behind moderation). '
            'Note: for borderline content, we may want to suppress (push to lower ranks) rather than '
            'hard-block, with user age-gating. This is a separate moderation threshold per content category.'
        ),
    },
    {
        'question': 'Q3: The model was trained 6 months ago. How do you know when to retrain?',
        'weak': 'We would retrain every few months on a schedule.',
        'strong': (
            'Monitor three signals for model staleness: '
            '(1) Embedding distribution drift: compute the mean and variance of embeddings from '
            'new images vs. training-era images. KL divergence above threshold triggers retraining. '
            '(2) Offline metric degradation: re-evaluate nDCG on a held-out test set monthly. '
            'More than 2 percent absolute drop is a trigger. '
            '(3) Online metric degradation: if CTR on visual search results drops significantly '
            '(detected by SPC/control charts), investigate whether it is a model issue. '
            'In practice, Pinterest retrains their visual search model quarterly with a mix of '
            'self-supervised pre-training and fine-tuning on fresh interaction data.'
        ),
    },
    {
        'question': 'Q4: How would you extend this system to support text-to-image search?',
        'weak': 'We would train a separate text embedding model and do the same ANN search.',
        'strong': (
            'Use a joint embedding space like CLIP (Contrastive Language-Image Pretraining from OpenAI). '
            'CLIP trains an image encoder and a text encoder jointly, so the embedding space is shared -- '
            'a text query "red vintage bicycle" produces an embedding close to images of red vintage bicycles. '
            'This shares the same FAISS index for both image and text queries, reducing infrastructure cost. '
            'The tradeoff: CLIP visual embeddings are slightly worse than pure image-trained models '
            'for image-to-image similarity. You might use CLIP for text queries and a fine-tuned '
            'ResNet for image queries, both searching the same index.'
        ),
    },
    {
        'question': 'Q5: How would you add personalization to the results?',
        'weak': 'We would add user features to the model and retrain with user IDs.',
        'strong': (
            'Personalization should be added in the re-ranking stage, not the embedding model. '
            'Here is why: the embedding model produces a single fixed representation per image. '
            'Baking user ID into the model would require a separate embedding per user -- impossible at scale. '
            'Instead: (1) Build a lightweight taste model per user based on their interaction history '
            '-- a weighted average of embeddings they have clicked. '
            '(2) In re-ranking, boost candidates whose embeddings are close to the user taste vector. '
            '(3) This also enables visual taste profiles without any model retraining. '
            'The ANN search stays universal; personalization is a cheap re-ranking step.'
        ),
    },
    {
        'question': 'Q6: How would you handle the cold-start problem for new images with zero interactions?',
        'weak': 'New images would be added to the index when they are uploaded.',
        'strong': (
            'Cold-start affects both discovery and relevance. '
            'For discovery: new images are embedded and added to the index incrementally -- '
            'they become searchable in under 5 minutes with no user interactions needed. '
            'The embedding model generalizes to new images without interaction data. '
            'For re-ranking and relevance scoring: we might slightly boost very new images '
            '(freshness bonus) to ensure they get some impressions, then let CTR data accumulate. '
            'This is a classic explore-exploit problem -- we are exploring new images to gather signal. '
            'The opposite problem is rich-get-richer: popular images always surface because they '
            'have high CTR. Combat this with impression-based normalization in the re-ranking score.'
        ),
    },
]

fig, axes = plt.subplots(len(qna_pairs), 1, figsize=(18, 36))

for i, (qna, ax) in enumerate(zip(qna_pairs, axes)):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 3.2)
    ax.axis('off')
    
    # Question background
    q_bg = mpatches.FancyBboxPatch((0, 2.7), 10, 0.5, boxstyle='round,pad=0.04',
                                    facecolor='#37474F', edgecolor='none', alpha=0.9)
    ax.add_patch(q_bg)
    ax.text(0.15, 2.96, qna['question'], fontsize=11, fontweight='bold', 
            color='white', va='center')
    
    # Weak answer box
    w_bg = mpatches.FancyBboxPatch((0, 1.35), 4.85, 1.3, boxstyle='round,pad=0.06',
                                    facecolor='#FFEBEE', edgecolor='#EF9A9A', linewidth=2)
    ax.add_patch(w_bg)
    ax.text(0.15, 2.5, 'WEAK ANSWER', fontsize=10, fontweight='bold', color='#C62828', va='top')
    wrapped_w = textwrap.fill(qna['weak'], width=48)
    ax.text(0.15, 2.28, wrapped_w, fontsize=9, color='#B71C1C', va='top')
    
    # Strong answer box
    s_bg = mpatches.FancyBboxPatch((5.15, 1.35), 4.85, 1.3, boxstyle='round,pad=0.06',
                                    facecolor='#E8F5E9', edgecolor='#81C784', linewidth=2)
    ax.add_patch(s_bg)
    ax.text(5.3, 2.5, 'STRONG ANSWER', fontsize=10, fontweight='bold', color='#1B5E20', va='top')
    wrapped_s = textwrap.fill(qna['strong'], width=55)
    ax.text(5.3, 2.28, wrapped_s, fontsize=9, color='#1B5E20', va='top')
    
    # Divider
    ax.axvline(5.1, ymin=0.42, ymax=1.0, color='#9E9E9E', linewidth=1.5, linestyle=':')
    
    # "vs" label
    ax.text(5.05, 2.0, 'vs', ha='center', fontsize=12, fontweight='bold', color='#9E9E9E')

fig.suptitle('6 Follow-up Questions: WEAK vs STRONG Answers', 
             fontsize=17, fontweight='bold', y=1.003)
plt.tight_layout()
plt.savefig('followup_qa.png', dpi=110, bbox_inches='tight')
plt.show()

## 6. Scoring Rubric

### How Interviewers Actually Assess You

Most companies use a rubric similar to this. Understanding the rubric lets you explicitly hit the criteria.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# ======================================================================
# Scoring Rubric: Junior / Senior / Staff Level Indicators
# ======================================================================

rubric = {
    'Dimension': [
        'Problem Clarification',
        'ML Framing',
        'Data Pipeline',
        'Model Architecture',
        'Training Strategy',
        'Evaluation Metrics',
        'Serving Architecture',
        'Scale Analysis',
        'Follow-up Depth',
        'Communication',
    ],
    'Junior': [
        'Asks 1-2 questions, may miss key ones (scale)',
        'Says "classification" or vague "deep learning"',
        'Mentions preprocessing exists',
        'Names ResNet, no details on projection head or L2 norm',
        'Says "train with labeled data"',
        'Mentions accuracy or MSE',
        'Describes one pipeline with rough boxes',
        'No numbers mentioned',
        'Generic answers, does not engage with the specifics',
        'Long pauses, uses filler words, cannot whiteboard',
    ],
    'Senior': [
        'Asks 4-5 focused questions, summarizes correctly',
        'Representation learning + ranking framing',
        'Preprocessing steps named, augmentation described',
        'ResNet-50 + projection head + L2 normalization',
        'Self-supervised contrastive (SimCLR), NT-Xent loss',
        'nDCG correctly justified over precision/recall',
        'Both pipelines, mentions ANN + re-ranking',
        'Order-of-magnitude estimates (200TB, O(N×D))',
        'Handles most follow-ups with reasonable answers',
        'Clear structure, able to whiteboard',
    ],
    'Staff': [
        'Asks questions AND signals constraints they imply (e.g., "200B means ANN")',
        'Explicit decoupling: offline embed + online search; mentions ViT vs ResNet tradeoff',
        'Domain-specific augmentation choices; IPS for debiasing click data',
        'Differential LR fine-tuning; alignment + uniformity evaluation',
        'MoCo vs SimCLR tradeoff by GPU memory; hard negative mining details',
        'nDCG@10 as headline; alignment/uniformity as proxy; A/B with Bonferroni correction',
        'PQ compression numbers; HNSW vs IVFFlat tradeoff; zero-downtime index swap',
        'Exact numbers: 200TB → 6TB with PQ, 1000 GPUs, nprobe=128, 95% recall',
        'Proactively raises follow-ups before asked; knows edge cases',
        'Draws whiteboard in real-time, structures answer in 5 phases, manages time',
    ],
}

df = pd.DataFrame(rubric)

fig, ax = plt.subplots(figsize=(24, 14))
ax.axis('off')

col_colors = ['#ECEFF1'] + ['#FFEBEE', '#E3F2FD', '#E8F5E9']
col_widths = [0.12, 0.25, 0.29, 0.34]

table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    loc='center',
    cellLoc='left',
)

table.auto_set_font_size(False)
table.set_fontsize(7.5)
table.auto_set_column_width(col=list(range(len(df.columns))))

# Header style
header_colors = ['#455A64', '#C62828', '#1565C0', '#1B5E20']
for col, (color, width) in enumerate(zip(header_colors, col_widths)):
    cell = table[0, col]
    cell.set_facecolor(color)
    cell.set_text_props(color='white', fontweight='bold', fontsize=9)

# Row styles
row_colors_data = ['#ECEFF1', '#FFCDD2', '#BBDEFB', '#C8E6C9']
for row in range(1, len(df)+1):
    for col in range(len(df.columns)):
        cell = table[row, col]
        base_color = row_colors_data[col]
        # Alternate rows slightly
        if row % 2 == 0:
            r, g, b = int(base_color[1:3], 16), int(base_color[3:5], 16), int(base_color[5:7], 16)
            darken = 0.95
            hex_color = f'#{int(r*darken):02x}{int(g*darken):02x}{int(b*darken):02x}'
            cell.set_facecolor(hex_color)
        else:
            cell.set_facecolor(base_color)
        cell.set_text_props(fontsize=7)
        
        # Make column 0 bold
        if col == 0:
            cell.set_text_props(fontweight='bold', fontsize=8)

table.scale(1, 2.0)  # Make rows taller for readability

ax.set_title('Scoring Rubric: Junior / Senior / Staff Level Indicators\n'
             'Visual Search System Design Interview',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('scoring_rubric.png', dpi=110, bbox_inches='tight')
plt.show()

print("Focus area: The top 3 dimensions that move you from Senior -> Staff:")
print("  1. Scale Analysis (put exact numbers on everything)")
print("  2. Serving Architecture (PQ compression, index swap, nprobe tuning)")
print("  3. Follow-up Depth (proactively raise edge cases)")

## 7. The 30-Second Elevator Pitch

If you had to explain this entire system in 30 seconds — in an elevator with the CEO — what would you say?

Practice this until it is effortless. It is also a great opening statement after the interviewer finishes the prompt.

---

### The Pitch (Memorize This)

> "We design a visual search system that maps every image to a 256-dimensional embedding using a contrastive-trained ResNet-50, then builds a FAISS IVFFlat index over all 200 billion embeddings — compressed to 6 TB with product quantization. At query time, we embed the query image on GPU in 15ms, run approximate nearest neighbor search in 10ms to get 500 candidates, then apply a re-ranking service that filters by privacy, deduplicates, and enforces diversity to return the top 50 results in under 100ms total. The offline indexing pipeline nightly rebuilds the index with 1,000 GPUs in 30 minutes, while incremental updates make new images searchable within 5 minutes."

**Word count:** ~95 words. ~30 seconds at a normal speaking pace.

**What it covers:**
- Model architecture (ResNet-50, 256-D, contrastive)
- Storage optimization (PQ compression, 6 TB)
- Prediction pipeline (GPU embed -> ANN -> re-rank, < 100ms)
- Indexing pipeline (nightly + incremental)
- Key numbers (1000 GPUs, 30 min rebuild, 5 min freshness)

---

### Memory Hook

Break it into 4 chunks:
1. **Model:** "contrastive ResNet-50, 256-D embeddings"
2. **Index:** "FAISS IVFFlat, PQ-compressed to 6 TB"
3. **Prediction:** "embed 15ms + ANN 10ms + re-rank = < 100ms"
4. **Indexing:** "nightly rebuild 30min + incremental 5min" 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ======================================================================
# Key Vocabulary Cheat Sheet
# Visual: color-coded by category for easy memorization
# ======================================================================

vocab = {
    "Embedding & Representation": [
        ("Embedding vector", "Fixed-dim dense representation of an image (e.g., 256-D float32)"),
        ("L2 normalization", "Scales vector to unit length; makes cosine sim = dot product"),
        ("Unit hypersphere", "The geometric surface where L2-normalized vectors live"),
        ("Contrastive learning", "Train by contrasting similar (positive) vs. dissimilar (negative) pairs"),
        ("NT-Xent loss", "Normalized Temperature-scaled Cross-Entropy; the SimCLR loss function"),
        ("Temperature tau", "Sharpness of softmax; 0.07 is the standard default for visual search"),
        ("Hard negatives", "Negatives visually similar to query -- force better representation learning"),
        ("Alignment + Uniformity", "Theoretical measures of embedding space quality (Wang & Isola 2020)"),
    ],
    "ANN Search": [
        ("ANN", "Approximate Nearest Neighbor -- trades recall for speed"),
        ("IVF (Inverted File Index)", "Cluster-based ANN; nlist clusters, search nprobe of them"),
        ("nlist", "Number of IVF clusters; rule of thumb: sqrt(N)"),
        ("nprobe", "Clusters searched per query; controls recall vs. speed"),
        ("Product Quantization", "Compress embeddings by encoding sub-vectors as centroid indices"),
        ("PQ32x8", "32 sub-quantizers x 8 bits = 32 bytes/vector (vs 1024 bytes float32)"),
        ("FAISS", "Meta AI open-source ANN library (CPU + GPU), industry standard"),
        ("ScaNN", "Google ANN library with anisotropic quantization, used in Google Search"),
        ("HNSW", "Hierarchical Navigable Small World -- best recall-latency curve, high memory"),
        ("O(D log N)", "Approximate complexity of tree/graph ANN methods vs O(N x D) brute force"),
    ],
    "Training & Data": [
        ("SimCLR", "Self-supervised contrastive learning via augmentation pairs (Google Brain 2020)"),
        ("MoCo", "Momentum Contrast -- uses a queue for more negatives at lower batch size (Meta 2020)"),
        ("Self-supervision", "No human labels; positive pairs created by augmenting the same image"),
        ("Positive pair", "Two images (or augmentations) labeled as similar for contrastive training"),
        ("IPS (Inverse Propensity Scoring)", "Technique to debias click data for positional effects"),
        ("Differential LR", "Use lower LR for pre-trained backbone, higher for new projection head"),
    ],
    "Serving": [
        ("Prediction pipeline", "Real-time: preprocess -> embed -> ANN search -> re-rank"),
        ("Indexing pipeline", "Offline: batch embed all images -> build FAISS index -> shard & serve"),
        ("Re-ranking service", "Applies business rules post-ANN: dedup, privacy, policy, diversity"),
        ("Positional bias", "Users click top results more regardless of quality; use IPS to correct"),
        ("Index swap", "Atomic switch from old to new index for zero-downtime model updates"),
        ("Incremental indexing", "Add new images to existing IVF clusters without full rebuild"),
    ],
}

n_categories = len(vocab)
category_colors = {
    "Embedding & Representation": "#B3E5FC",
    "ANN Search": "#C8E6C9",
    "Training & Data": "#FFE0B2",
    "Serving": "#F8BBD0",
}

fig, axes = plt.subplots(n_categories, 1, figsize=(22, 30))

for ax, (category, terms) in zip(axes, vocab.items()):
    color = category_colors[category]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, len(terms) + 0.7)
    ax.axis("off")
    
    # Category header
    hdr = mpatches.FancyBboxPatch((0, len(terms)), 10, 0.65, 
                                   boxstyle="round,pad=0.05", facecolor=color, 
                                   edgecolor="#555", linewidth=2)
    ax.add_patch(hdr)
    ax.text(0.2, len(terms) + 0.32, category, fontsize=13, fontweight="bold", va="center")
    
    for i, (term, definition) in enumerate(terms):
        y = len(terms) - 1 - i
        row_color = color if i % 2 == 0 else "#FAFAFA"
        row_bg = mpatches.FancyBboxPatch((0, y), 10, 0.9, boxstyle="round,pad=0.02",
                                         facecolor=row_color, edgecolor="none", alpha=0.3)
        ax.add_patch(row_bg)
        ax.text(0.2, y + 0.45, term, fontsize=10, fontweight="bold", va="center")
        ax.text(3.0, y + 0.45, definition, fontsize=9, va="center", color="#333")
        ax.axhline(y, color="#DDD", linewidth=0.5)

fig.suptitle("Key Vocabulary Cheat Sheet: Visual Search System Design",
             fontsize=17, fontweight="bold", y=1.004)
plt.tight_layout()
plt.savefig("vocabulary_cheatsheet.png", dpi=110, bbox_inches="tight")
plt.show()

print("Vocabulary coverage:", sum(len(v) for v in vocab.values()), "key terms across 4 categories.")
print("Print this and review it the night before your interview.")

## 8. Interview Preparation Checklist

### The Night Before

- [ ] Recite the 30-second elevator pitch without looking at notes
- [ ] Draw both architecture diagrams (prediction + indexing) freehand in < 5 minutes
- [ ] Compute the back-of-envelope: storage for 200B images with float32 and PQ
- [ ] Review the scoring rubric: identify which dimensions you're strongest/weakest on
- [ ] Practice explaining IVF (nlist, nprobe, the recall-speed knob)

### During the Interview

- [ ] Write the 5-phase outline on the whiteboard at the start: Clarify | Frame | Data+Model | Eval | Serving
- [ ] Ask exactly 4-5 clarifying questions — no more, no less
- [ ] State numbers: "256-dimensional embeddings", "nprobe=128", "6 TB with PQ32x8"
- [ ] When the interviewer probes, ask "Is this the depth you want?" — manage time actively
- [ ] At the end: "I have more to say about X — should I go deeper there?"

### Key Numbers to Know Cold

| Metric | Number |
|--------|--------|
| Embedding dimension | 256 (or 512 for ViT) |
| Float32 storage for 200B × 256 | ~200 TB |
| PQ32x8 storage for 200B | ~6 TB |
| Compression ratio | ~32× |
| IVF clusters (nlist) | sqrt(N) ≈ 65,536 for 4B (shards of 200B) |
| nprobe for 95% recall | ~128 |
| ANN latency target | < 10ms (CPU) |
| End-to-end latency target | < 100ms |
| GPU throughput (ResNet-50, A100) | ~500 imgs/sec |
| GPUs for 30-min nightly rebuild | ~1,000 A100s |
| Index freshness (new images) | < 5 minutes |
| Temperature (τ) | 0.07 |
| Batch size for SimCLR | 4096+ |

---

*This completes the Visual Search ML Design module. You are now ready for a staff-level system design interview on this topic.*

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)